In [901]:
from dataclasses import dataclass

@dataclass
class Grid:
    rows: int
    cols: int
    step_reward: int
    terminals: dict
    walls: set
    actions = dict(up=(-1, 0), right=(0, 1), down=(1, 0), left=(0, -1))
    arrows = {'up': "↑", 'right': "→", 'down': "↓", 'left': "←"}
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return self.terminals[cell]
        elif cell in self.walls:
            return '#'
        else:
            return '·'
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def step(self, cell, action):
        movement = self.actions[action]
        next_cell = (cell[0] + movement[0], cell[1] + movement[1])
        outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
        on_wall = next_cell in self.walls
        if outside or on_wall:
            next_cell = cell

        if next_cell in self.terminals:
            reward = self.terminals[next_cell]
        else:
            reward = self.step_reward
        return next_cell, reward
    def properties(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
)
'''


default_grid = Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)
default_grid

Grid(rows=3, cols=4, step_reward=0, terminals={(0, 3): 1}, walls={(1, 1)})

In [902]:
print(default_grid.properties())


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)



In [903]:
default_grid.render()

  ·   ·   ·   1 
  ·   #   ·   · 
  ·   ·   ·   · 


In [904]:
cell = (0, 0)
for _ in range(3):
    cell, reward = default_grid.step(cell, 'right')
    print((cell, reward))


((0, 1), 0)
((0, 2), 0)
((0, 3), 1)


In [905]:
def show_V(grid: Grid, V):
    for r in range(grid.rows):
        for c in range(grid.cols):
            print(f'{round(V[r][c], 2):>4} ', end="")
        print()
    print('------------------')

    
def value_iteration(grid: Grid, gamma = 0.9):
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = 1e9
    while delta > 0.001:
        delta = 0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                new_value = float('-inf')
                for action in grid.actions:
                    next_cell, reward = grid.step(cell, action)
                    new_value = max(
                        new_value, reward + gamma * V_old[next_cell[0]][next_cell[1]]
                    )
                V[r][c] = new_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        show_V(grid, V)
    return V

In [906]:
V = value_iteration(default_grid)

 0.0  0.0  1.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.0 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0  0.0  0.9 
------------------
0.81  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------


In [907]:
def read_policy(grid: Grid, V, gamma = 0.9):
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            cell = (r, c)
            if cell in grid.walls or cell in grid.terminals:
                continue

            max_value = float('-inf')
            for action in grid.actions:
                next_cell, reward = grid.step(cell, action)
                action_value = reward + gamma * V[next_cell[0]][next_cell[1]]
                if action_value > max_value:
                    policy[r][c] = action
                    max_value= action_value
    return policy

policy = read_policy(default_grid, V)
policy

[['right', 'right', 'right', None],
 ['up', None, 'up', 'up'],
 ['up', 'right', 'up', 'up']]

In [908]:
def render_policy(grid: Grid, policy):
    for r in range(len(policy)):
        for c in range(len(policy[0])):
            if policy[r][c] != None:
                value = grid.arrows[policy[r][c]]
            else:
                value = grid.cell_repr(r, c)
            print(f' {value} ', end='')
        print()
    print('------------------')
render_policy(default_grid, policy)

 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------


In [909]:
def policy_evaluation(grid: Grid, policy, gamma = 0.9, verbose = False):
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = 1e9
    while delta > 0.001:
        delta = 0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                action = policy[r][c]
                next_cell, reward = grid.step(cell, action)
                V[r][c] = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            show_V(grid, V)
    return V

V = policy_evaluation(default_grid, policy, verbose=True)

 0.0  0.0  1.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.0 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0  0.0  0.9 
------------------
0.81  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------


In [910]:
def policy_iteration(grid: Grid):
    policy = [['up' for _ in range(grid.cols)] for _ in range(grid.rows)]
    while True:
        V = policy_evaluation(grid, policy)
        show_V(grid, V)
        new_policy = read_policy(grid, V)
        changed = sum(
            new_policy[r][c] != None and new_policy[r][c] != policy[r][c]
            for r in range(grid.rows)
            for c in range(grid.cols)
        )
        print(f'actions changed = {changed}')
        render_policy(grid, new_policy)
        if new_policy == policy:
            break
        policy = new_policy
    return policy, V

In [911]:
policy, V = policy_iteration(default_grid)

 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.9 
------------------
actions changed = 3
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
actions changed = 4
 ↑  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
actions changed = 2
 →  →  →  1 
 ↑  #  ↑  ↑ 
 →  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 1
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 0
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------


In [912]:
def show_V(grid: Grid, V):
    for r in range(grid.rows):
        for c in range(grid.cols):
            print(f'{round(V[r][c], 2):>4} ', end="")
        print()
    print('------------------')


def read_policy(grid: Grid, V, gamma = 0.9, p_policy = None):
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            cell = (r, c)
            if cell in grid.walls or cell in grid.terminals:
                continue
            if p_policy:
                p_action = p_policy[r][c]
                p_next_cell, p_reward = grid.step(cell, p_action)
                p_action_value = p_reward + gamma * V[p_next_cell[0]][p_next_cell[1]]
            
            max_value = float('-inf')
            for action in grid.actions:
                next_cell, reward = grid.step(cell, action)
                action_value = reward + gamma * V[next_cell[0]][next_cell[1]]
                if action_value > max_value:
                    if p_policy and action_value - p_action_value < 1e-9:
                        action = p_action
                        action_value = p_action_value
                    policy[r][c] = action
                    max_value= action_value
    return policy


def render_policy(grid: Grid, policy):
    for r in range(len(policy)):
        for c in range(len(policy[0])):
            if policy[r][c] != None:
                value = grid.arrows[policy[r][c]]
            else:
                value = grid.cell_repr(r, c)
            print(f' {value} ', end='')
        print()
    print('------------------')


def value_iteration(grid: Grid, gamma = 0.9, max_iters = 1000):
    print('---------- value iteration ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = 1e9
    i = 0
    while delta > 0.001 and i < max_iters:
        delta = 0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                new_value = float('-inf')
                for action in grid.actions:
                    next_cell, reward = grid.step(cell, action)
                    new_value = max(
                        new_value, reward + gamma * V_old[next_cell[0]][next_cell[1]]
                    )
                V[r][c] = new_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        show_V(grid, V)
        i += 1
    
    policy = read_policy(grid, V)
    render_policy(grid, policy)
    converged = i < max_iters
    if not converged:
        print(f'value_iteration did not converge in {max_iters} iterations')
    return V


def policy_evaluation(grid: Grid, policy, gamma = 0.9, verbose = False, max_iters=1000):
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = 1e9
    i = 0
    while delta > 0.001 and i < max_iters:
        delta = 0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                action = policy[r][c]
                next_cell, reward = grid.step(cell, action)
                V[r][c] = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            show_V(grid, V)
        i += 1
    converged = i < max_iters
    if not converged:
        print(f'policy_evaluation did not converge in {max_iters} iterations')
    return V, converged


def policy_iteration(
    grid: Grid,
    gamma=0.9,
    max_iters = 1000,
    pass_incumbent_policy=False,
    policy_evaluation_max_iters=1000,
    policy_evaluation_verbose=False,
):
    print('---------- policy iteration ------------')
    policy = [
        ["up" if grid.cell_repr(r, c) == "·" else None for c in range(grid.cols)]
        for r in range(grid.rows)
    ]
    policy_evaluation_converged = True
    i = 0
    while i < max_iters:
        V, policy_evaluation_converged = policy_evaluation(
            grid,
            policy,
            gamma=gamma,
            max_iters=policy_evaluation_max_iters,
            verbose=policy_evaluation_verbose,
        )
        if not policy_evaluation_converged:
            break
        show_V(grid, V)
        new_policy = read_policy(
            grid,
            V,
            gamma=gamma,
            p_policy=policy if pass_incumbent_policy else None,
        )
        changed = sum(
            new_policy[r][c] != policy[r][c]
            for r in range(grid.rows)
            for c in range(grid.cols)
        )
        print(f'actions changed = {changed}')
        render_policy(grid, new_policy)
        i += 1
        if new_policy == policy:
            break
        policy = new_policy

    converged = i < max_iters
    if converged:
        if policy_evaluation_converged:
            print(f'policy_iteration converged in {i} iterations')
        else:
            print(
                f"policy_iteration did not converge because policy_evaluation did not converge"
            )
    else:
        print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged and policy_evaluation_converged

In [913]:
grid = Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)

In [914]:
print(grid.properties())
value_iteration(grid);


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)

---------- value iteration ------------
 0.0  0.0  1.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.0 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0  0.0  0.9 
------------------
0.81  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------


In [915]:
print(grid.properties())
policy_iteration(grid);


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)

---------- policy iteration ------------
 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.9 
------------------
actions changed = 3
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
actions changed = 4
 ↑  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
actions changed = 2
 →  →  →  1 
 ↑  #  ↑  ↑ 
 →  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 1
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 0
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
policy_iteration converged in 5 iterations


In [916]:
print(grid.properties())
policy_iteration(grid, pass_incumbent_policy=True);


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)

---------- policy iteration ------------
 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.9 
------------------
actions changed = 3
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
actions changed = 2
 ↑  →  →  1 
 ↑  #  →  ↑ 
 ↑  →  →  ↑ 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
actions changed = 2
 →  →  →  1 
 ↑  #  →  ↑ 
 →  →  →  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 0
 →  →  →  1 
 ↑  #  →  ↑ 
 →  →  →  ↑ 
------------------
policy_iteration converged in 4 iterations


In [917]:
grid = Grid(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): 0},
    walls={(1, 1)},
)

In [918]:
print(grid.properties())
value_iteration(grid);


Grid(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): 0},
    walls={(1, 1)},
)

---------- value iteration ------------
-1.0 -1.0  0.0    0 
-1.0    0 -1.0  0.0 
-1.0 -1.0 -1.0 -1.0 
------------------
-1.9 -1.0  0.0    0 
-1.9    0 -1.0  0.0 
-1.9 -1.9 -1.9 -1.0 
------------------
-1.9 -1.0  0.0    0 
-2.71    0 -1.0  0.0 
-2.71 -2.71 -1.9 -1.0 
------------------
-1.9 -1.0  0.0    0 
-2.71    0 -1.0  0.0 
-3.44 -2.71 -1.9 -1.0 
------------------
-1.9 -1.0  0.0    0 
-2.71    0 -1.0  0.0 
-3.44 -2.71 -1.9 -1.0 
------------------
 →  →  →  0 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------


In [919]:
print(grid.properties())
policy_iteration(grid);


Grid(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): 0},
    walls={(1, 1)},
)

---------- policy iteration ------------
-9.99 -9.99 -9.99    0 
-9.99    0 -9.99  0.0 
-9.99 -9.99 -9.99 -1.0 
------------------
actions changed = 3
 ↑  ↑  →  0 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
-9.99 -9.99  0.0    0 
-9.99    0 -1.0  0.0 
-9.99 -9.99 -1.9 -1.0 
------------------
actions changed = 4
 ↑  →  →  0 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
-9.99 -1.0  0.0    0 
-9.99    0 -1.0  0.0 
-9.99 -2.71 -1.9 -1.0 
------------------
actions changed = 2
 →  →  →  0 
 ↑  #  ↑  ↑ 
 →  →  ↑  ↑ 
------------------
-1.9 -1.0  0.0    0 
-2.71    0 -1.0  0.0 
-3.44 -2.71 -1.9 -1.0 
------------------
actions changed = 1
 →  →  →  0 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
-1.9 -1.0  0.0    0 
-2.71    0 -1.0  0.0 
-3.44 -2.71 -1.9 -1.0 
------------------
actions changed = 0
 →  →  →  0 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
policy_iteration converged in 5 ite

In [920]:
grid = Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)
gamma = 1.0

In [921]:
print(grid.properties())
print(f'{gamma=}')
value_iteration(grid, gamma=gamma);


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)

gamma=1.0
---------- value iteration ------------
 0.0  0.0  1.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.0 
------------------
 0.0  1.0  1.0    0 
 0.0    0  1.0  1.0 
 0.0  0.0  0.0  1.0 
------------------
 1.0  1.0  1.0    0 
 0.0    0  1.0  1.0 
 0.0  0.0  1.0  1.0 
------------------
 1.0  1.0  1.0    0 
 1.0    0  1.0  1.0 
 0.0  1.0  1.0  1.0 
------------------
 1.0  1.0  1.0    0 
 1.0    0  1.0  1.0 
 1.0  1.0  1.0  1.0 
------------------
 1.0  1.0  1.0    0 
 1.0    0  1.0  1.0 
 1.0  1.0  1.0  1.0 
------------------
 ↑  ↑  →  1 
 ↑  #  ↑  ↑ 
 ↑  ↑  ↑  ↑ 
------------------


In [922]:
print(grid.properties())
print(f'{gamma=}')
policy_iteration(grid, max_iters=100, gamma=gamma);


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)

gamma=1.0
---------- policy iteration ------------
 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  1.0 
------------------
actions changed = 3
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  1.0  1.0 
 0.0  0.0  1.0  1.0 
------------------
actions changed = 5
 ↑  →  ↑  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  1.0 
------------------
actions changed = 5
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  1.0  1.0 
 0.0  0.0  1.0  1.0 
------------------
actions changed = 5
 ↑  →  ↑  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  1.0 
------------------
actions changed = 5
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  1.0  1.0 
 0.0  0

In [923]:
print(grid.properties())
print(f'{gamma=}')
policy_iteration(grid, gamma=gamma, pass_incumbent_policy=True);


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)

gamma=1.0
---------- policy iteration ------------
 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  1.0 
------------------
actions changed = 3
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  1.0  1.0 
 0.0  0.0  1.0  1.0 
------------------
actions changed = 2
 ↑  →  →  1 
 ↑  #  →  ↑ 
 ↑  →  →  ↑ 
------------------
 0.0  1.0  1.0    0 
 0.0    0  1.0  1.0 
 0.0  1.0  1.0  1.0 
------------------
actions changed = 2
 →  →  →  1 
 ↑  #  →  ↑ 
 →  →  →  ↑ 
------------------
 1.0  1.0  1.0    0 
 1.0    0  1.0  1.0 
 1.0  1.0  1.0  1.0 
------------------
actions changed = 0
 →  →  →  1 
 ↑  #  →  ↑ 
 →  →  →  ↑ 
------------------
policy_iteration converged in 4 iterations


In [924]:
grid = Grid(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): -1},
    walls={(1, 1)},
)
gamma = 1.0

In [925]:
print(grid.properties())
print(f'{gamma=}')
value_iteration(grid, gamma=gamma);


Grid(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): -1},
    walls={(1, 1)},
)

gamma=1.0
---------- value iteration ------------
-1.0 -1.0 -1.0    0 
-1.0    0 -1.0 -1.0 
-1.0 -1.0 -1.0 -1.0 
------------------
-2.0 -2.0 -1.0    0 
-2.0    0 -2.0 -1.0 
-2.0 -2.0 -2.0 -2.0 
------------------
-3.0 -2.0 -1.0    0 
-3.0    0 -2.0 -1.0 
-3.0 -3.0 -3.0 -2.0 
------------------
-3.0 -2.0 -1.0    0 
-4.0    0 -2.0 -1.0 
-4.0 -4.0 -3.0 -2.0 
------------------
-3.0 -2.0 -1.0    0 
-4.0    0 -2.0 -1.0 
-5.0 -4.0 -3.0 -2.0 
------------------
-3.0 -2.0 -1.0    0 
-4.0    0 -2.0 -1.0 
-5.0 -4.0 -3.0 -2.0 
------------------
 →  →  →  -1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------


In [926]:
print(grid.properties())
print(f'{gamma=}')
policy_iteration(
        grid,
        gamma=gamma,
        policy_evaluation_max_iters=100,
        policy_evaluation_verbose=True,
    );


Grid(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): -1},
    walls={(1, 1)},
)

gamma=1.0
---------- policy iteration ------------
-1.0 -1.0 -1.0    0 
-1.0    0 -1.0 -1.0 
-1.0 -1.0 -1.0 -1.0 
------------------
-2.0 -2.0 -2.0    0 
-2.0    0 -2.0 -1.0 
-2.0 -2.0 -2.0 -2.0 
------------------
-3.0 -3.0 -3.0    0 
-3.0    0 -3.0 -1.0 
-3.0 -3.0 -3.0 -2.0 
------------------
-4.0 -4.0 -4.0    0 
-4.0    0 -4.0 -1.0 
-4.0 -4.0 -4.0 -2.0 
------------------
-5.0 -5.0 -5.0    0 
-5.0    0 -5.0 -1.0 
-5.0 -5.0 -5.0 -2.0 
------------------
-6.0 -6.0 -6.0    0 
-6.0    0 -6.0 -1.0 
-6.0 -6.0 -6.0 -2.0 
------------------
-7.0 -7.0 -7.0    0 
-7.0    0 -7.0 -1.0 
-7.0 -7.0 -7.0 -2.0 
------------------
-8.0 -8.0 -8.0    0 
-8.0    0 -8.0 -1.0 
-8.0 -8.0 -8.0 -2.0 
------------------
-9.0 -9.0 -9.0    0 
-9.0    0 -9.0 -1.0 
-9.0 -9.0 -9.0 -2.0 
------------------
-10.0 -10.0 -10.0    0 
-10.0    0 -10.0 -1.0 
-10.0 -10.0 -10.0 -2.0 
------------------
-11.0 -11.0 -11.0  

In [927]:
grid = Grid(
    rows=5,
    cols=6,
    step_reward=-1,
    terminals={(0, 5): 0},
    walls={(2, 2), (2, 3), (3, 1), (3, 4), (4, 2), (4, 3)},
)
grid.render()

  ·   ·   ·   ·   ·   0 
  ·   ·   ·   ·   ·   · 
  ·   ·   #   #   ·   · 
  ·   #   ·   ·   #   · 
  ·   ·   #   #   ·   · 


In [928]:
print(grid.properties())
value_iteration(grid);


Grid(
    rows=5,
    cols=6,
    step_reward=-1,
    terminals={(0, 5): 0},
    walls={(2, 3), (2, 2), (3, 4), (4, 3), (3, 1), (4, 2)},
)

---------- value iteration ------------
-1.0 -1.0 -1.0 -1.0  0.0    0 
-1.0 -1.0 -1.0 -1.0 -1.0  0.0 
-1.0 -1.0    0    0 -1.0 -1.0 
-1.0    0 -1.0 -1.0    0 -1.0 
-1.0 -1.0    0    0 -1.0 -1.0 
------------------
-1.9 -1.9 -1.9 -1.0  0.0    0 
-1.9 -1.9 -1.9 -1.9 -1.0  0.0 
-1.9 -1.9    0    0 -1.9 -1.0 
-1.9    0 -1.9 -1.9    0 -1.9 
-1.9 -1.9    0    0 -1.9 -1.9 
------------------
-2.71 -2.71 -1.9 -1.0  0.0    0 
-2.71 -2.71 -2.71 -1.9 -1.0  0.0 
-2.71 -2.71    0    0 -1.9 -1.0 
-2.71    0 -2.71 -2.71    0 -1.9 
-2.71 -2.71    0    0 -2.71 -2.71 
------------------
-3.44 -2.71 -1.9 -1.0  0.0    0 
-3.44 -3.44 -2.71 -1.9 -1.0  0.0 
-3.44 -3.44    0    0 -1.9 -1.0 
-3.44    0 -3.44 -3.44    0 -1.9 
-3.44 -3.44    0    0 -3.44 -2.71 
------------------
-3.44 -2.71 -1.9 -1.0  0.0    0 
-4.1 -3.44 -2.71 -1.9 -1.0  0.0 
-4.1 -4.1    0    0 -1.9 -1.0

In [929]:
print(grid.properties())
policy_iteration(grid);


Grid(
    rows=5,
    cols=6,
    step_reward=-1,
    terminals={(0, 5): 0},
    walls={(2, 3), (2, 2), (3, 4), (4, 3), (3, 1), (4, 2)},
)

---------- policy iteration ------------
-9.99 -9.99 -9.99 -9.99 -9.99    0 
-9.99 -9.99 -9.99 -9.99 -9.99  0.0 
-9.99 -9.99    0    0 -9.99 -1.0 
-9.99    0 -9.99 -9.99    0 -1.9 
-9.99 -9.99    0    0 -9.99 -2.71 
------------------
actions changed = 4
 ↑  ↑  ↑  ↑  →  0 
 ↑  ↑  ↑  ↑  →  ↑ 
 ↑  ↑  #  #  →  ↑ 
 ↑  #  ↑  ↑  #  ↑ 
 ↑  ↑  #  #  →  ↑ 
------------------
-9.99 -9.99 -9.99 -9.99  0.0    0 
-9.99 -9.99 -9.99 -9.99 -1.0  0.0 
-9.99 -9.99    0    0 -1.9 -1.0 
-9.99    0 -9.99 -9.99    0 -1.9 
-9.99 -9.99    0    0 -3.44 -2.71 
------------------
actions changed = 4
 ↑  ↑  ↑  →  →  0 
 ↑  ↑  ↑  →  ↑  ↑ 
 ↑  ↑  #  #  ↑  ↑ 
 ↑  #  ↑  ↑  #  ↑ 
 ↑  ↑  #  #  →  ↑ 
------------------
-9.99 -9.99 -9.99 -1.0  0.0    0 
-9.99 -9.99 -9.99 -1.9 -1.0  0.0 
-9.99 -9.99    0    0 -1.9 -1.0 
-9.99    0 -9.99 -9.99    0 -1.9 
-9.99 -9.99    0    0 -3.44 -2

In [930]:
gamma = 1.0

In [931]:
print(grid.properties())
print(f'{gamma=}')
value_iteration(grid, gamma=gamma, max_iters=100);


Grid(
    rows=5,
    cols=6,
    step_reward=-1,
    terminals={(0, 5): 0},
    walls={(2, 3), (2, 2), (3, 4), (4, 3), (3, 1), (4, 2)},
)

gamma=1.0
---------- value iteration ------------
-1.0 -1.0 -1.0 -1.0  0.0    0 
-1.0 -1.0 -1.0 -1.0 -1.0  0.0 
-1.0 -1.0    0    0 -1.0 -1.0 
-1.0    0 -1.0 -1.0    0 -1.0 
-1.0 -1.0    0    0 -1.0 -1.0 
------------------
-2.0 -2.0 -2.0 -1.0  0.0    0 
-2.0 -2.0 -2.0 -2.0 -1.0  0.0 
-2.0 -2.0    0    0 -2.0 -1.0 
-2.0    0 -2.0 -2.0    0 -2.0 
-2.0 -2.0    0    0 -2.0 -2.0 
------------------
-3.0 -3.0 -2.0 -1.0  0.0    0 
-3.0 -3.0 -3.0 -2.0 -1.0  0.0 
-3.0 -3.0    0    0 -2.0 -1.0 
-3.0    0 -3.0 -3.0    0 -2.0 
-3.0 -3.0    0    0 -3.0 -3.0 
------------------
-4.0 -3.0 -2.0 -1.0  0.0    0 
-4.0 -4.0 -3.0 -2.0 -1.0  0.0 
-4.0 -4.0    0    0 -2.0 -1.0 
-4.0    0 -4.0 -4.0    0 -2.0 
-4.0 -4.0    0    0 -4.0 -3.0 
------------------
-4.0 -3.0 -2.0 -1.0  0.0    0 
-5.0 -4.0 -3.0 -2.0 -1.0  0.0 
-5.0 -5.0    0    0 -2.0 -1.0 
-5.0    0 -5.0 -5.0 

In [932]:
print(grid.properties())
print(f'{gamma=}')
policy_iteration(grid, gamma=gamma, policy_evaluation_verbose=True, policy_evaluation_max_iters=100);


Grid(
    rows=5,
    cols=6,
    step_reward=-1,
    terminals={(0, 5): 0},
    walls={(2, 3), (2, 2), (3, 4), (4, 3), (3, 1), (4, 2)},
)

gamma=1.0
---------- policy iteration ------------
-1.0 -1.0 -1.0 -1.0 -1.0    0 
-1.0 -1.0 -1.0 -1.0 -1.0  0.0 
-1.0 -1.0    0    0 -1.0 -1.0 
-1.0    0 -1.0 -1.0    0 -1.0 
-1.0 -1.0    0    0 -1.0 -1.0 
------------------
-2.0 -2.0 -2.0 -2.0 -2.0    0 
-2.0 -2.0 -2.0 -2.0 -2.0  0.0 
-2.0 -2.0    0    0 -2.0 -1.0 
-2.0    0 -2.0 -2.0    0 -2.0 
-2.0 -2.0    0    0 -2.0 -2.0 
------------------
-3.0 -3.0 -3.0 -3.0 -3.0    0 
-3.0 -3.0 -3.0 -3.0 -3.0  0.0 
-3.0 -3.0    0    0 -3.0 -1.0 
-3.0    0 -3.0 -3.0    0 -2.0 
-3.0 -3.0    0    0 -3.0 -3.0 
------------------
-4.0 -4.0 -4.0 -4.0 -4.0    0 
-4.0 -4.0 -4.0 -4.0 -4.0  0.0 
-4.0 -4.0    0    0 -4.0 -1.0 
-4.0    0 -4.0 -4.0    0 -2.0 
-4.0 -4.0    0    0 -4.0 -3.0 
------------------
-5.0 -5.0 -5.0 -5.0 -5.0    0 
-5.0 -5.0 -5.0 -5.0 -5.0  0.0 
-5.0 -5.0    0    0 -5.0 -1.0 
-5.0    0 -5.0 -5.0

In [933]:
grid = Grid(
    rows=3,
    cols=4,
    step_reward=-0.04,          # the knob to sweep
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)
grid.render()

  ·   ·   ·   1 
  ·   #   ·  -1 
  ·   ·   ·   · 


In [934]:
print(grid.properties())
value_iteration(grid);


Grid(
    rows=3,
    cols=4,
    step_reward=-0.04,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

---------- value iteration ------------
-0.04 -0.04  1.0    0 
-0.04    0 -0.04    0 
-0.04 -0.04 -0.04 -0.04 
------------------
-0.08 0.86  1.0    0 
-0.08    0 0.86    0 
-0.08 -0.08 -0.08 -0.08 
------------------
0.73 0.86  1.0    0 
-0.11    0 0.86    0 
-0.11 -0.11 0.73 -0.11 
------------------
0.73 0.86  1.0    0 
0.62    0 0.86    0 
-0.14 0.62 0.73 0.62 
------------------
0.73 0.86  1.0    0 
0.62    0 0.86    0 
0.52 0.62 0.73 0.62 
------------------
0.73 0.86  1.0    0 
0.62    0 0.86    0 
0.52 0.62 0.73 0.62 
------------------
 →  →  →  1 
 ↑  #  ↑  -1 
 ↑  →  ↑  ← 
------------------


In [ ]:
grid = Grid(
    rows=3,
    cols=4,
    step_reward=-1,          # the knob to sweep
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)
grid.render()

  ·   ·   ·   1 
  ·   #   ·  -1 
  ·   ·   ·   · 


In [938]:
print(grid.properties())
value_iteration(grid);


Grid(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

---------- value iteration ------------
-1.0 -1.0  1.0    0 
-1.0    0 -1.0    0 
-1.0 -1.0 -1.0 -1.0 
------------------
-1.9 -0.1  1.0    0 
-1.9    0 -0.1    0 
-1.9 -1.9 -1.9 -1.0 
------------------
-1.09 -0.1  1.0    0 
-2.71    0 -0.1    0 
-2.71 -2.71 -1.09 -1.0 
------------------
-1.09 -0.1  1.0    0 
-1.98    0 -0.1    0 
-3.44 -1.98 -1.09 -1.0 
------------------
-1.09 -0.1  1.0    0 
-1.98    0 -0.1    0 
-2.78 -1.98 -1.09 -1.0 
------------------
-1.09 -0.1  1.0    0 
-1.98    0 -0.1    0 
-2.78 -1.98 -1.09 -1.0 
------------------
 →  →  →  1 
 ↑  #  ↑  -1 
 ↑  →  ↑  ↑ 
------------------


The threshold is exactly -1.9, and you can check it by hand:


```-
q(1,2 | UP)    = s + 0.9 * V(0,2) = s + 0.9 * 1 = s + 0.9   # detour to the +1
q(1,2 | RIGHT) = -1                                          # step into the trap, done
```

So RIGHT wins once -1 > s + 0.9, i.e. s < -1.9. Below that, (1,2) and its neighbours flip from ↑ to → — the agent commits suicide because living costs more than the penalty for dying. Above it, the arrows route around the trap.

In [ ]:
grid = Grid(
    rows=3,
    cols=4,
    step_reward=-2,          # the knob to sweep
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)
grid.render()

  ·   ·   ·   1 
  ·   #   ·  -1 
  ·   ·   ·   · 


In [940]:
print(grid.properties())
value_iteration(grid);


Grid(
    rows=3,
    cols=4,
    step_reward=-2,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

---------- value iteration ------------
-2.0 -2.0  1.0    0 
-2.0    0 -1.0    0 
-2.0 -2.0 -2.0 -1.0 
------------------
-3.8 -1.1  1.0    0 
-3.8    0 -1.0    0 
-3.8 -3.8 -2.9 -1.0 
------------------
-2.99 -1.1  1.0    0 
-5.42    0 -1.0    0 
-5.42 -4.61 -2.9 -1.0 
------------------
-2.99 -1.1  1.0    0 
-4.69    0 -1.0    0 
-6.15 -4.61 -2.9 -1.0 
------------------
-2.99 -1.1  1.0    0 
-4.69    0 -1.0    0 
-6.15 -4.61 -2.9 -1.0 
------------------
 →  →  →  1 
 ↑  #  →  -1 
 →  →  ↑  ↑ 
------------------
